# colab_06 — Second integration method (scANVI) + scIB comparison

Sixth notebook. colab_04 integrated the three studies with **scVI** (label-free) and saved the
integrated object (`scvi_integrated_full.h5ad`, 694,922 cells / 145 donors) plus the trained scVI
model. This notebook runs the **second integration method, scANVI**, warm-started from that scVI
model, and produces the head-to-head **scIB** comparison (competency-spec item 2: "a second
integration method, evaluate both with scIB, articulate the tradeoffs").

**Why scANVI is the second method (locked 2026-05-21).** scVI corrects on `study_id`. But study is
**confounded with APOE** — the studies were not assembled with matched APOE distributions, so
correcting away study-of-origin risks also flattening the APOE-carrier axis that eval #2 depends
on. **scANVI is label-aware** (semi-supervised): it is given coarse cell-type labels and corrects
batch *while being told what biological structure to keep*. The load-bearing question this notebook
answers is therefore not just "which mixes the studies better" but **"does scANVI preserve the APOE
axis within each glia lineage better than scVI, after study correction?"** (§6b).

**Strategy (carried from the integration locks):**
- **ITS — integrate-then-subset.** Like scVI, scANVI harmonizes at the whole-tissue level on
  **coarse lineage labels** (astrocyte / microglia / oligodendrocyte / OPC / neuron / vascular).
  Within-glia substate structure is **not** imposed here — that is the foundation model's job, and
  feeding substate labels would pre-bake the very structure eval #1 later tests. Labels stay coarse.
- **Warm start.** `SCANVI.from_scvi_model` initializes from the saved scVI weights, so scANVI only
  refines the latent rather than retraining from scratch (~20 epochs, not 150).
- **`batch_key = study_id`** (inherited from the scVI model registry), `labels_key = cell_type`.

**The labels.** colab_05 assigned a coarse `cell_type` to every cell (marker-score argmax, agreeing
with the colab_04 territory map on 14/14 anchor clusters) but saved **only the glia subset**. The
full integrated object on Drive carries `leiden_scvi` but not `cell_type`, so §4 **rebuilds** the
coarse labels with the identical procedure (deterministic — it reproduces colab_05's annotation),
and fails loud if the anchor clusters diverge.

**scIB fairness caveat (reported, not hidden).** scANVI is *trained toward* the cell-type labels,
so any bio-conservation edge it shows in scIB is partly expected by construction. The discriminating
comparisons are therefore (i) the **batch-correction** metrics, where neither method has a label
advantage, and (ii) the **within-lineage APOE recovery** in §6b. Both methods are scored on the
**same** labels and the **same** stratified subsample so the comparison is apples-to-apples.

**What this notebook does NOT do (deferred):**
- The astro/micro subset already exists (colab_05). cl44 weak-margin microglia → micro-subset notebook.
- The FM continued-pretraining + the three pre-committed evals (`EVALUATION_CONTRACT.md`) → FM notebooks.

**Runtime: requires a GPU runtime (A100).** Integration env (`requirements_integration.txt`,
Py3.12, scvi-tools 1.4.3). The scIB benchmark (§6a) is the slow CPU-bound step — run on a stratified
subsample.

## 1 — Setup

### 1a — Mount Drive + clone/pull repo + install env

Identical pattern to `colab_04`: mount Drive, clone-or-pull the repo, pin numpy first, install the
integration env. Adds an explicit `scib-metrics` install (the §6a benchmark) in case it is not
pinned in `requirements_integration.txt`.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels (same rationale as colab_01-04).
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt
# scIB metrics for the scVI-vs-scANVI comparison (§6a); ensure present even if unpinned.
!pip install scib-metrics

## 2 — Environment capture

### 2a — pip freeze + env JSON

Snapshot exact versions to `outputs/software_versions/` (paper Methods). Same structure as colab_04;
`scib_metrics` is the addition that matters here.

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_06"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":   NOTEBOOK_ID,
    "date":          TODAY,
    "python_version": sys.version,
    "platform":      platform.platform(),
    "os_release":    platform.release(),
    "gpu":           _run(["nvidia-smi", "-L"]),
    "nvidia_driver": _run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]),
    "git_commit":    _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scvi_tools_version": _ver("scvi"),
    "scanpy_version":     _ver("scanpy"),
    "anndata_version":    _ver("anndata"),
    "jax_version":        _ver("jax"),
    "scib_metrics_version": _ver("scib_metrics"),
}
try:
    import torch
    env_snapshot["torch_version"]      = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
    env_snapshot["cuda_available"]     = bool(torch.cuda.is_available())
    env_snapshot["cudnn_version"]      = torch.backends.cudnn.version() if torch.cuda.is_available() else None
except ImportError:
    env_snapshot["torch_version"]  = None
    env_snapshot["cuda_available"] = None
    env_snapshot["cudnn_version"]  = None

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Load the integrated object + scVI model

scANVI does not start from raw data — it starts from what scVI already learned. This section reloads
the integrated object (full-gene raw counts plus the scVI latent `X_scVI`) and confirms it still
carries the two things scANVI needs: the `X_scVI` latent and the `in_scvi` flag marking the ~3,000
genes scVI was trained on (the scVI model has to be reloaded onto **exactly** that feature matrix).

### 3a — Load `scvi_integrated_full.h5ad`, sanity-check latent + feature flag + raw counts

Load the colab_04 hand-off object. Fail loud unless `X_scVI` (the scVI latent) and `var["in_scvi"]`
(the scVI feature set) are present and `.X` is raw counts (scANVI, like scVI, models counts). Also
set up the per-notebook figure directory.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

FIG_DIR = os.path.join(REPO_PATH, "figures", "colab_06")
os.makedirs(FIG_DIR, exist_ok=True)

INTEGRATED_PATH = os.path.join(DRIVE_ROOT, "integrated", "scvi_integrated_full.h5ad")
if not os.path.exists(INTEGRATED_PATH):
    raise FileNotFoundError(f"missing colab_04 hand-off object {INTEGRATED_PATH}")
adata = sc.read_h5ad(INTEGRATED_PATH)
print("loaded:", adata.shape)

assert "X_scVI" in adata.obsm, "X_scVI missing — colab_04 hand-off object expected"
assert "in_scvi" in adata.var.columns, "in_scvi gene flag missing — needed to rebuild the scVI feature matrix"
assert "leiden_scvi" in adata.obs.columns, "leiden_scvi missing — needed to rebuild cell_type labels"

# raw-counts guard: random 2000-cell sample (sorted deposits hide non-integer tails in a head slice)
_idx = np.random.default_rng(0).choice(adata.n_obs, size=min(2000, adata.n_obs), replace=False)
Xs = adata.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}); scANVI requires counts"

# stash the scVI UMAP before scANVI overwrites obsm['X_umap'] in 5b
if "X_umap" in adata.obsm and "X_umap_scvi" not in adata.obsm:
    adata.obsm["X_umap_scvi"] = adata.obsm["X_umap"].copy()

print(f"X_scVI {adata.obsm['X_scVI'].shape} | in_scvi genes {int(adata.var['in_scvi'].sum())} "
      f"| leiden clusters {adata.obs['leiden_scvi'].nunique()} | raw-counts int-frac {frac_int:.3f}")
print("study mix:", adata.obs["study_id"].value_counts(normalize=True).round(3).to_dict())
_ram("loaded integrated")

## 4 — Rebuild coarse cell-type labels

scANVI needs a label per cell. colab_05 produced those labels but saved only the glia subset, so the
full object loaded above has none. This section **reconstructs** them with colab_05's exact procedure
— score every cell against canonical marker panels, assign each Leiden cluster to its highest-scoring
lineage — so the labels reproduce colab_05's annotation rather than inventing a new one. The labels
stay **coarse** (lineage level): scANVI's job is cross-study lineage harmonization, not substate
structure.

### 4a — Marker-score argmax per cluster → `cell_type` (reproduces colab_05 4b)

Build a log-normalized layer (`.X` stays raw counts), score each cell per lineage signature, take the
per-cluster mean, and assign each cluster its argmax lineage. Carry colab_05's one adjudicated
override (cluster 0 = perivascular macrophage, not microglia). **Fail-loud anchor check:** the
colab_04/05 territory-map anchor clusters must map to the same lineage, else the rebuild diverged and
the scANVI labels are untrustworthy.

In [ ]:
# log-normalized layer for scoring; .X stays raw counts throughout
adata.layers["lognorm"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4, layer="lognorm")
adata.uns.pop("log1p", None)            # stale guard would make layer-level log1p silently skip
sc.pp.log1p(adata, layer="lognorm")

# canonical markers per major brain cell type (colab_05 MARKERS, verbatim)
MARKERS = {
    "astrocyte":        ["AQP4", "SLC1A2", "GFAP", "GJA1", "SLC1A3"],
    "microglia":        ["CSF1R", "P2RY12", "TMEM119", "AIF1", "TREM2", "CX3CR1"],
    "oligodendrocyte":  ["MBP", "MOBP", "PLP1", "MOG"],
    "OPC":              ["PDGFRA", "CSPG4", "OLIG1", "OLIG2"],
    "neuron_exc":       ["RBFOX3", "SNAP25", "SLC17A7", "SATB2"],
    "neuron_inh":       ["GAD1", "GAD2"],
    "endothelial":      ["CLDN5", "FLT1", "PECAM1"],
    "mural":            ["PDGFRB", "RGS5", "DCN"],
    "perivascular_mac": ["MRC1", "CD163", "LYVE1", "F13A1"],
}
present = {ct: [g for g in gs if g in adata.var_names] for ct, gs in MARKERS.items()}
for ct, gs in present.items():
    miss = set(MARKERS[ct]) - set(gs)
    if miss:
        print(f"[markers] {ct}: absent from gene set -> {sorted(miss)}")

score_cols = []
X_raw = adata.X
adata.X = adata.layers["lognorm"]
try:
    for ct, gs in present.items():
        if gs:
            sc.tl.score_genes(adata, gene_list=gs, score_name=f"score_{ct}", use_raw=False)
            score_cols.append(f"score_{ct}")
finally:
    adata.X = X_raw                      # ALWAYS restore raw counts

S = adata.obs.groupby("leiden_scvi", observed=True)[score_cols].mean()
S.columns = [c.replace("score_", "") for c in S.columns]
assign = S.idxmax(axis=1)
srt = np.sort(S.values, axis=1)
margin = pd.Series(srt[:, -1] - srt[:, -2], index=S.index).round(3)
print("per-cluster mean marker scores:")
with pd.option_context("display.max_rows", None, "display.width", 160):
    print(S.round(2))
    print("\nassignment (sorted by margin):")
    print(pd.DataFrame({"cell_type": assign, "margin": margin}).sort_values("margin"))

adata.obs["cell_type"] = adata.obs["leiden_scvi"].map(assign).astype(str)
# colab_05 adjudication carried forward: cluster 0 = perivascular macrophage (NOT microglia)
adata.obs.loc[adata.obs["leiden_scvi"] == "0", "cell_type"] = "perivascular_mac"

# fail-loud anchor check vs the colab_04 / colab_05 territory map
ANCHORS = {
    "microglia":       ["1"],
    "astrocyte":       ["22", "28", "30", "31", "33", "42", "45"],
    "oligodendrocyte": ["16", "17", "18", "19", "21", "35"],
}
mismatch = [(cl, ct, assign.get(cl, "MISSING"))
            for ct, cls in ANCHORS.items() for cl in cls if assign.get(cl, "MISSING") != ct]
if mismatch:
    print("\n[ANCHOR MISMATCH] rebuild diverged from colab_05 — scANVI labels untrustworthy:")
    for cl, exp, got in mismatch:
        print(f"  cl{cl}: expected {exp}, got {got}")
    raise AssertionError(f"{len(mismatch)} anchor cluster(s) diverged from colab_05 annotation")
print(f"\nanchor check: all {sum(len(v) for v in ANCHORS.values())} anchor clusters match colab_05")
assert adata.obs["cell_type"].isin(["nan", "None"]).sum() == 0, "unmapped cells — every cluster must get a label"

print("\ncell_type counts (full object):")
print(adata.obs["cell_type"].value_counts())
print("\ncell_type x study:")
print(pd.crosstab(adata.obs["cell_type"], adata.obs["study_id"]))
_ram("after relabel")

## 5 — scANVI integration

scANVI takes scVI's already-learned latent space and refines it with one extra piece of information:
the coarse cell-type label. Because it starts from the scVI weights (warm start), it needs only a
short fine-tune. The result is a second latent space, `X_scANVI`, that is label-guided where scVI's
was label-free — the two are then compared head-to-head in §6.

### 5a — Warm-start `SCANVI.from_scvi_model` + train

Rebuild the exact feature matrix scVI trained on (`in_scvi` genes, raw counts), reload the saved scVI
model onto it, then initialize scANVI from those weights with `labels_key="cell_type"`. An
`unlabeled_category="Unknown"` is declared for the API even though every cell here is labeled. Short
fine-tune (`max_epochs=20`) since the model is warm-started; `n_samples_per_label=100` balances the
label classes during training. **Fails loud without a GPU.**

In [ ]:
import torch, scvi
import matplotlib.pyplot as plt

scvi.settings.seed = 0
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. scANVI needs a GPU runtime (A100). "
                       "Switch runtime type to GPU and re-run from 1a.")
print("GPU:", torch.cuda.get_device_name(0))

# exact scVI feature matrix: in_scvi genes, raw counts, with cell_type carried onto obs
adata_hvg = adata[:, adata.var["in_scvi"]].copy()
adata_hvg.obs["cell_type"] = adata.obs["cell_type"].values

SCVI_MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "scvi_colab04")
scvi_model = scvi.model.SCVI.load(SCVI_MODEL_DIR, adata=adata_hvg)
print("reloaded scVI model from", SCVI_MODEL_DIR)

scanvi_model = scvi.model.SCANVI.from_scvi_model(
    scvi_model,
    adata=adata_hvg,
    labels_key="cell_type",
    unlabeled_category="Unknown",
)
scanvi_model.train(
    max_epochs=20,
    n_samples_per_label=100,
    early_stopping=True,
    early_stopping_patience=10,
    check_val_every_n_epoch=1,
)

hist = scanvi_model.history
fig, ax = plt.subplots(figsize=(6, 4))
for k, lab in [("elbo_train", "train"), ("elbo_validation", "validation")]:
    if k in hist:
        hist[k].plot(ax=ax, label=lab)
ax.set_xlabel("epoch"); ax.set_ylabel("ELBO"); ax.legend(); ax.set_title("scANVI training")
plt.show()
print("epochs trained:", len(hist.get("elbo_train", [])))
_ram("after scANVI train")

### 5b — scANVI latent + neighbors + Leiden + UMAP

Write the scANVI latent onto the full object as `obsm["X_scANVI"]`, then build a separate neighbor
graph / Leiden / UMAP on it (stored under scANVI-specific keys) so §6c can show scVI and scANVI side
by side without one overwriting the other.

In [ ]:
adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation(adata_hvg)
print("scANVI latent:", adata.obsm["X_scANVI"].shape)
del adata_hvg; gc.collect()

sc.pp.neighbors(adata, use_rep="X_scANVI", n_neighbors=15, key_added="scanvi_nn")
sc.tl.leiden(adata, resolution=1.0, key_added="leiden_scanvi",
             flavor="igraph", n_iterations=2, directed=False, neighbors_key="scanvi_nn")
sc.tl.umap(adata, min_dist=0.3, neighbors_key="scanvi_nn")
adata.obsm["X_umap_scanvi"] = adata.obsm["X_umap"].copy()
print("scANVI Leiden clusters:", adata.obs["leiden_scanvi"].nunique())
_ram("after scANVI umap")

## 6 — scVI vs scANVI comparison

Three views, all on the same cells. **6a** is the standard scIB scorecard (batch correction vs
bio-conservation). **6b** is the project-specific question scANVI was chosen to answer — does it
preserve the APOE axis within glia better than scVI after study correction. **6c** is the qualitative
side-by-side UMAP. Read 6a's *batch* metrics and 6b as the discriminating evidence; 6a's *bio* metrics
are biased toward scANVI (it was trained on those labels) and are reported with that caveat.

### 6a — scIB metrics (batch + bio) for both latents

`scib_metrics.Benchmarker` scores both embeddings on batch correction (iLISI, kBET, graph
connectivity, batch-ASW) and bio conservation (NMI/ARI/cell-type-ASW/isolated-label) against
`cell_type`. Run on a **study × cell_type stratified subsample** (the full 694k object is intractable
for kBET/silhouette) — both embeddings are scored on the identical subsample. Results saved to CSV for
the write-up.

In [ ]:
from scib_metrics.benchmark import Benchmarker

# stratified subsample so the slow scIB metrics are tractable; both embeddings see the SAME cells
N_BENCH = 100_000
rng = np.random.default_rng(0)
strata = (adata.obs["study_id"].astype(str) + "|" + adata.obs["cell_type"].astype(str)).values
frac = min(1.0, N_BENCH / adata.n_obs)
idx_all = np.arange(adata.n_obs)
sel = np.concatenate([
    rng.choice(idx_all[strata == s],
               size=max(1, int(round((strata == s).sum() * frac))), replace=False)
    for s in pd.unique(strata)
])
bench = adata[np.sort(sel)].copy()
print("scIB benchmark subsample:", bench.shape)

bm = Benchmarker(
    bench,
    batch_key="study_id",
    label_key="cell_type",
    embedding_obsm_keys=["X_scVI", "X_scANVI"],
    n_jobs=-1,
)
bm.benchmark()
scib_df = bm.get_results(min_max_scale=False)
print(scib_df)

SCIB_CSV = os.path.join(REPO_PATH, "outputs", f"scib_scvi_vs_scanvi_{TODAY}.csv")
scib_df.to_csv(SCIB_CSV)
try:
    bm.plot_results_table(min_max_scale=False, show=True)
    plt.savefig(os.path.join(FIG_DIR, "scib_results_table.png"), dpi=150, bbox_inches="tight")
except Exception as e:
    print("plot_results_table skipped:", e)

### 6b — APOE × study confound + within-lineage E4 recovery (the load-bearing check)

First expose the confound: the APOE-genotype × study contingency, and the E4-carrier fraction per
study (if one study is mostly carriers, study correction can leak into APOE structure). Then the
decisive comparison — within **astrocytes** and within **microglia** separately, how recoverable is
binary E4-carrier status from each latent (silhouette + 5-fold kNN balanced accuracy)? scANVI is the
2nd method precisely so this can be asked.

**Confound caveat carried from colab_05 (cl33 `real_confounded`):** APOE recovery here can ride on a
small set of donor-pure clusters plus residual ambient, not a genuine population-level axis. This §6b
result is a **diagnostic** on the integration latents — it is *not* eval #2 (which is on the FM
embeddings, per `EVALUATION_CONTRACT.md`); both must report the study/region breakdown of the cells
driving any recovery.

In [ ]:
from sklearn.metrics import silhouette_score, balanced_accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold

# (i) the confound this 2nd-method choice exists to expose
print("APOE genotype x study (cell counts):")
print(pd.crosstab(adata.obs["apoe_genotype"], adata.obs["study_id"]))
print("\nE4-carrier fraction within each study:")
carr = adata.obs["apoe_carrier"].astype(str).str.lower()
y_all = np.where(carr.isin(["carrier", "true", "1", "yes"]), "carrier",
        np.where(carr.isin(["noncarrier", "non_carrier", "false", "0", "no"]), "non_carrier", "unknown"))
print(pd.crosstab(pd.Series(y_all, index=adata.obs.index, name="E4"),
                  adata.obs["study_id"], normalize="columns").round(3))

# (ii) within-lineage E4 recoverability, scVI vs scANVI
def e4_recovery(emb, y):
    sil = float(silhouette_score(emb, y, sample_size=min(10000, len(y)), random_state=0))
    accs = []
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(emb, y):
        knn = KNeighborsClassifier(n_neighbors=15).fit(emb[tr], y[tr])
        accs.append(balanced_accuracy_score(y[te], knn.predict(emb[te])))
    return sil, float(np.mean(accs))

rows = []
valid = y_all != "unknown"
for lineage in ["astrocyte", "microglia"]:
    mask = (adata.obs["cell_type"].values == lineage) & valid
    y = y_all[mask]
    if len(np.unique(y)) < 2:
        print(f"{lineage}: single E4 class present, skipped"); continue
    # subsample to 40k for tractability; SAME indices across both embeddings
    if mask.sum() > 40000:
        si = np.sort(np.random.default_rng(0).choice(np.where(mask)[0], 40000, replace=False))
    else:
        si = np.where(mask)[0]
    y_s = y_all[si]
    for rep in ["X_scVI", "X_scANVI"]:
        sil, acc = e4_recovery(adata.obsm[rep][si], y_s)
        rows.append({"lineage": lineage, "embedding": rep, "n_cells": int(mask.sum()),
                     "silhouette_E4": round(sil, 4), "knn_baccuracy_E4": round(acc, 4)})

res = pd.DataFrame(rows)
print("\nWithin-lineage E4-carrier recoverability (higher = APOE axis better preserved):")
print(res.to_string(index=False))
APOE_CSV = os.path.join(REPO_PATH, "outputs", f"apoe_recovery_scvi_vs_scanvi_{TODAY}.csv")
res.to_csv(APOE_CSV, index=False)
_ram("after apoe recovery")

### 6c — Side-by-side UMAPs (study / cell_type / APOE)

The qualitative read: the scVI UMAP (`X_umap_scvi`) and the scANVI UMAP (`X_umap_scanvi`) coloured by
study, lineage, and E4-carrier status. Good integration = each lineage one study-mixed region;
the APOE panel is for eyeballing whether carrier status survives as visible structure or is washed
out.

In [ ]:
for color in ["study_id", "cell_type", "apoe_carrier"]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (rep, basis) in zip(axes, [("scVI", "umap_scvi"), ("scANVI", "umap_scanvi")]):
        if f"X_{basis}" not in adata.obsm:
            ax.set_visible(False); continue
        sc.pl.embedding(adata, basis=basis, color=color, ax=ax, show=False,
                        title=f"{rep} — {color}", frameon=False, size=2)
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"umap_compare_{color}.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 7 — Save scANVI model + latent

Storage discipline (Audit F): the 6 GB full object is **not** re-saved. Persist a compact latent
**sidecar** (obs + both latents + both UMAPs) for any downstream comparison, plus the trained scANVI
model so it can be reloaded without retraining.

### 7a — Save the latent sidecar to Drive

A small AnnData holding the key obs columns and both integration latents (`X_scVI`, `X_scANVI`) +
both UMAPs — a few hundred MB, vs re-writing the full 6 GB object. Drive-only (gitignored).

In [ ]:
SIDE_OBS = ["donor_id", "sample_id", "study_id", "region", "apoe_genotype", "apoe_carrier",
            "diagnosis", "cell_type", "leiden_scvi", "leiden_scanvi"]
side = ad.AnnData(
    X=sp.csr_matrix((adata.n_obs, 1), dtype="float32"),
    obs=adata.obs[[c for c in SIDE_OBS if c in adata.obs.columns]].copy(),
)
side.obs_names = adata.obs_names
for k in ["X_scVI", "X_scANVI", "X_umap_scvi", "X_umap_scanvi"]:
    if k in adata.obsm:
        side.obsm[k] = adata.obsm[k]

LATENT_PATH = os.path.join(DRIVE_ROOT, "integrated", "scanvi_latent_sidecar.h5ad")
side.write_h5ad(LATENT_PATH)
print(f"saved latent sidecar -> {LATENT_PATH}  ({os.path.getsize(LATENT_PATH)/1e6:.0f} MB)")
print("  obsm:", list(side.obsm.keys()))

### 7b — Save the trained scANVI model

Save the scANVI model directory (`save_anndata=False` — data is on Drive via 7a / the colab_04
object).

In [ ]:
SCANVI_MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "scanvi_colab06")
os.makedirs(os.path.dirname(SCANVI_MODEL_DIR), exist_ok=True)
scanvi_model.save(SCANVI_MODEL_DIR, overwrite=True, save_anndata=False)
print("scANVI model saved to", SCANVI_MODEL_DIR)

## 8 — Handoff

Record the scANVI run into the cumulative `audit_report.json`, **flip Audit B** (SEA-AD + Li donor
APOE were `skipped` pending integration — now confirmed present and integrated), sort artifacts into
committable vs Drive-only, and print the manual WSL commit commands.

### 8a — Write scANVI trace + Audit B flip + commit instructions

Append an `integration_scanvi` entry (warm-start source, label key, scIB subsample size, result CSV
paths, cell-type counts) and update `audit_b_apoe_metadata` SEA-AD/Li entries to `pass`. Committable:
env freeze/JSON, `audit_report.json`, the scIB + APOE-recovery CSVs. Drive-only: latent sidecar +
scANVI model.

In [ ]:
import shlex

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

report["integration_scanvi"] = {
    "status": "computed",
    "date": TODAY,
    "warm_start_from": "scvi_colab04",
    "batch_key": "study_id",
    "labels_key": "cell_type",
    "n_cells": int(adata.n_obs),
    "n_scanvi_leiden_clusters": int(adata.obs["leiden_scanvi"].nunique()),
    "scib_subsample_n": int(bench.n_obs),
    "scib_results_csv": os.path.relpath(SCIB_CSV, REPO_PATH),
    "apoe_recovery_csv": os.path.relpath(APOE_CSV, REPO_PATH),
    "cell_type_counts": adata.obs["cell_type"].value_counts().to_dict(),
}

# Audit B flip: SEA-AD + Li were 'skipped' (path is None) pending integration confirmation.
ab = report.get("audit_b_apoe_metadata", {})
for entry in ab.get("per_study", []):
    if entry.get("study") in ("SEA-AD", "Li 2025") and entry.get("status") == "skipped":
        entry["status"] = "pass"
        entry["note"] = "donor-level APOE confirmed present + integrated (colab_06)"
        s = "SEA-AD" if entry["study"] == "SEA-AD" else "Li2025"
        m = adata.obs["study_id"].astype(str) == s
        entry["n_donors_with_apoe"] = int(adata.obs.loc[m, "donor_id"].nunique())
if all(e.get("status") == "pass" for e in ab.get("per_study", [])):
    ab["status"] = "pass"
report["audit_b_apoe_metadata"] = ab

with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("=== Artifacts ===")
print("  committable: ", FREEZE_PATH)
print("  committable: ", ENV_JSON_PATH)
print("  committable: ", AUDIT_REPORT_PATH)
print("  committable: ", SCIB_CSV)
print("  committable: ", APOE_CSV)
print("  drive-only : ", LATENT_PATH)
print("  drive-only : ", SCANVI_MODEL_DIR)

rel = [os.path.relpath(p, REPO_PATH) for p in
       (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH, SCIB_CSV, APOE_CSV)]
print("\n=== Commit + push (from WSL — Colab has no git creds) ===")
print("  cd /content/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /content/ad-glia-fm-prep && git commit -m "
      "'colab_06: scANVI 2nd integration + scIB comparison run record'")
print("  cd /content/ad-glia-fm-prep && git push")

### Carried forward to the FM notebooks

- **2nd-method verdict (written into the _OUTPUT after the run):** read §6a *batch* metrics + §6b APOE
  recovery to decide whether scANVI's label-awareness buys preservation of the APOE axis that scVI's
  study correction costs — the articulated scVI-vs-scANVI tradeoff (competency item 2).
- **Substrate is unchanged.** CPT runs on the colab_05 glia subset (raw genes), not on either
  integration latent. The latents + this comparison inform interpretation, not the FM input.
- **APOE confound (cl33 `real_confounded`) travels into eval #2:** any FM-embedding APOE recovery must
  report the study/region breakdown of the driving cells, exactly as §6b does here.
- **cl44** weak-margin microglia → resolved as cell 1 of the micro-subset notebook (Option C).